In [33]:
import pandas as pd
import joblib
import os
from google.colab import drive
drive.mount('/content/drive')
ruta_train = '/content/drive/My Drive/Training_And_Testing_Dataset/smoking_prediction_processed.csv'
df_train = pd.read_csv(ruta_train)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [102]:
# ========================================================
# 04_entrenamiento_RF - Randomforest
# ========================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
import joblib
import os

# 1. Preparación de variables (asumiendo que X_tr, X_te, y_tr, y_te ya están creados)
# Si no, usa: X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Entrenamiento con Regularización
# max_depth=10 evita que los árboles crezcan sin límite (causa #1 de overfitting en RF)
rf_modelo = RandomForestClassifier(
    n_estimators=150,       # Un poco más de árboles para mayor estabilidad
    max_depth=7,           # Limite de profundidad para obligar a generalizar
    min_samples_leaf=10,     # Obliga a que cada hoja tenga al menos 5 casos (reduce ruido)
    random_state=42,
    n_jobs=-1               # Usa todos los núcleos de tu PC para ir más rápido
)

rf_modelo.fit(X_tr, y_tr)

# 3. Evaluación comparativa
pred_tr_rf = rf_modelo.predict(X_tr)
pred_te_rf = rf_modelo.predict(X_te)

f1_tr_rf = f1_score(y_tr, pred_tr_rf)
f1_te_rf = f1_score(y_te, pred_te_rf)
diferencia_rf = f1_tr_rf - f1_te_rf

print("--- REPORTE RANDOM FOREST  ---")
print(f"F1-Score Entrenamiento: {f1_tr_rf:.4f}")
print(f"F1-Score Test:          {f1_te_rf:.4f}")
print(f"Brecha (Overfitting):   {diferencia_rf:.4f}")

if diferencia_rf < 0.05:
    print("✅ Resultado: Modelo generalizado correctamente.")
else:
    print("⚠️ Nota: Considera ajustar 'max_depth' o 'min_samples_leaf' para mejorar la generalización.")

    # 4. Guardado
ruta_modelos = '/content/drive/My Drive/modelos_smoking/'
os.makedirs(ruta_modelos, exist_ok=True)
ruta_rf = os.path.join(ruta_modelos, 'Randomforest.pkl')
joblib.dump(rf_modelo, ruta_rf)

print(f"\n🏆 Modelo Regresión Logística guardado en: {ruta_rf}")

--- REPORTE RANDOM FOREST  ---
F1-Score Entrenamiento: 0.6968
F1-Score Test:          0.6779
Brecha (Overfitting):   0.0189
✅ Resultado: Modelo generalizado correctamente.

🏆 Modelo Regresión Logística guardado en: /content/drive/My Drive/modelos_smoking/Randomforest.pkl


In [104]:
# ========================================================
# 04_entrenamiento_y_optimizacion - XGBOOST
# ========================================================
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from google.colab import drive

# 1. Configuración de entorno y carga de datos
drive.mount('/content/drive')
ruta_csv = '/content/drive/My Drive/Training_And_Testing_Dataset/smoking_prediction_processed.csv'

# Verificación de existencia de archivo
if not os.path.exists(ruta_csv):
    raise FileNotFoundError(f"No se encontró el archivo en: {ruta_csv}")

df = pd.read_csv(ruta_csv)

# 2. Preparación de variables
X = df.drop(columns=['ID', 'smoking'], errors='ignore')
y = df['smoking']

# 3. División con estratificación para mantener balance de clases
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Entrenamiento con Regularización (Ajustado para evitar sobreajuste)
# max_depth reducido y parámetros de subsample añadidos para generalizar
xgb_global = XGBClassifier(
    n_estimators=100,
    max_depth=4,            # Profundidad moderada para no memorizar
    learning_rate=0.1,
    subsample=0.8,          # Aleatoriedad en datos
    colsample_bytree=0.8,   # Aleatoriedad en variables
    eval_metric='logloss',
    random_state=42,
    use_label_encoder=False
)

xgb_global.fit(X_tr, y_tr)

# 5. Evaluación comparativa detallada
pred_tr = xgb_global.predict(X_tr)
pred_te = xgb_global.predict(X_te)

f1_tr = f1_score(y_tr, pred_tr)
f1_te = f1_score(y_te, pred_te)
diferencia = f1_tr - f1_te

print("--- REPORTE DE DESEMPEÑO---")
print(f"F1-Score Entrenamiento: {f1_tr:.4f}")
print(f"F1-Score Test:          {f1_te:.4f}")
print(f"Brecha (Overfitting):   {diferencia:.4f}")

if diferencia < 0.05:
    print("✅ Resultado: Modelo generalizado correctamente.")
else:
    print("⚠️ Nota: La brecha sigue siendo alta, se recomienda más limpieza de datos.")

    # 4. Guardado
ruta_modelos = '/content/drive/My Drive/modelos_smoking/'
os.makedirs(ruta_modelos, exist_ok=True)
ruta_xgb = os.path.join(ruta_modelos, 'XGBOOST.pkl')
joblib.dump(xgb_global, ruta_xgb)

print(f"\n🏆 Modelo Regresión Logística guardado en: {ruta_xgb}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [08:07:16] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


--- REPORTE DE DESEMPEÑO---
F1-Score Entrenamiento: 0.7118
F1-Score Test:          0.6922
Brecha (Overfitting):   0.0196
✅ Resultado: Modelo generalizado correctamente.

🏆 Modelo Regresión Logística guardado en: /content/drive/My Drive/modelos_smoking/XGBOOST.pkl


In [105]:
# ========================================================
# 04_entrenamiento_y_optimizacion - Adaboost con Validación
# ========================================================
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score

# 1. Definición del modelo
# Mantener max_depth=1 es fundamental para evitar memorización
modelo_adaboost = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=100,
    learning_rate=0.5, # Ajuste fino: un learning_rate más bajo ayuda a la generalización
    random_state=42
)

# 2. Entrenamiento
modelo_adaboost.fit(X_tr, y_tr)

# 3. Evaluación comparativa
pred_tr_ada = modelo_adaboost.predict(X_tr)
pred_te_ada = modelo_adaboost.predict(X_te)

f1_tr_ada = f1_score(y_tr, pred_tr_ada)
f1_te_ada = f1_score(y_te, pred_te_ada)
diferencia_ada = f1_tr_ada - f1_te_ada

print("--- REPORTE ADABOOST ---")
print(f"F1-Score Entrenamiento: {f1_tr_ada:.4f}")
print(f"F1-Score Test:          {f1_te_ada:.4f}")
print(f"Brecha (Overfitting):   {diferencia_ada:.4f}")

if diferencia_ada < 0.05:
    print("✅ Resultado: Modelo generalizado correctamente.")
else:
    print("⚠️ Nota: AdaBoost puede ser sensible. Ajusta 'n_estimators' o 'learning_rate'.")

    # 4. Guardado
ruta_modelos = '/content/drive/My Drive/modelos_smoking/'
os.makedirs(ruta_modelos, exist_ok=True)
ruta_ada = os.path.join(ruta_modelos, 'adaboost.pkl')
joblib.dump(modelo_adaboost, ruta_ada)

print(f"\n🏆 Modelo Regresión Logística guardado en: {ruta_ada}")


--- REPORTE ADABOOST ---
F1-Score Entrenamiento: 0.6778
F1-Score Test:          0.6675
Brecha (Overfitting):   0.0104
✅ Resultado: Modelo generalizado correctamente.

🏆 Modelo Regresión Logística guardado en: /content/drive/My Drive/modelos_smoking/adaboost.pkl


In [106]:
# ========================================================
# 04_entrenamiento_y_optimizacion - REGRESIÓN LOGÍSTICA
# ========================================================
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

# 1. Pipeline: Escalado + Regresión
# max_iter=2000 asegura convergencia; solver='lbfgs' es el estándar moderno
pipeline_log_final = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=2000, solver='lbfgs', random_state=42))
])

# 2. Entrenamiento
pipeline_log_final.fit(X_tr, y_tr)

# 3. Evaluación comparativa
pred_tr_log = pipeline_log_final.predict(X_tr)
pred_te_log = pipeline_log_final.predict(X_te)

f1_tr_log = f1_score(y_tr, pred_tr_log)
f1_te_log = f1_score(y_te, pred_te_log)
diferencia_log = f1_tr_log - f1_te_log

print("--- REPORTE REGRESIÓN LOGÍSTICA ---")
print(f"F1-Score Entrenamiento: {f1_tr_log:.4f}")
print(f"F1-Score Test:          {f1_te_log:.4f}")
print(f"Brecha:                 {diferencia_log:.4f}")

# 4. Guardado
ruta_modelos = '/content/drive/My Drive/modelos_smoking/'
os.makedirs(ruta_modelos, exist_ok=True)
ruta_mrl = os.path.join(ruta_modelos, 'reglogistica.pkl')
joblib.dump(pipeline_log_final, ruta_mrl)

print(f"\n🏆 Modelo Regresión Logística guardado en: {ruta_mrl}")

--- REPORTE REGRESIÓN LOGÍSTICA ---
F1-Score Entrenamiento: 0.6658
F1-Score Test:          0.6650
Brecha:                 0.0008

🏆 Modelo Regresión Logística guardado en: /content/drive/My Drive/modelos_smoking/reglogistica.pkl


In [107]:
# ========================================================
# 04_entrenamiento_y_optimizacion - KNN CON VALIDACIÓN
# ========================================================
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score

# 1. Pipeline ajustado (usamos n_neighbors=15 para mayor suavidad)
# Un número mayor de vecinos ayuda a generalizar mejor en datasets con ruido
pipeline_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=70, n_jobs=-1))
])

# 2. Entrenamiento
pipeline_knn.fit(X_tr, y_tr)

# 3. Evaluación comparativa
pred_tr_knn = pipeline_knn.predict(X_tr)
pred_te_knn = pipeline_knn.predict(X_te)

f1_tr_knn = f1_score(y_tr, pred_tr_knn)
f1_te_knn = f1_score(y_tr, pred_tr_knn) # Nota: comparamos train vs test
f1_te_knn_real = f1_score(y_te, pred_te_knn)
diferencia_knn = f1_tr_knn - f1_te_knn_real

print("--- REPORTE KNN ---")
print(f"F1-Score Entrenamiento: {f1_tr_knn:.4f}")
print(f"F1-Score Test:          {f1_te_knn_real:.4f}")
print(f"Brecha (Overfitting):   {diferencia_knn:.4f}")

if diferencia_knn < 0.05:
    print("✅ Resultado: Modelo generalizado correctamente.")
else:
    print("⚠️ Nota: Ajusta 'n_neighbors' (más vecinos = más generalización).")

    # 4. Guardado
ruta_modelos = '/content/drive/My Drive/modelos_smoking/'
os.makedirs(ruta_modelos, exist_ok=True)
ruta_knn = os.path.join(ruta_modelos, 'knn.pkl')
joblib.dump(pipeline_knn, ruta_knn)

print(f"\n🏆 Modelo Regresión Logística guardado en: {ruta_knn}")


--- REPORTE KNN ---
F1-Score Entrenamiento: 0.6669
F1-Score Test:          0.6457
Brecha (Overfitting):   0.0212
✅ Resultado: Modelo generalizado correctamente.

🏆 Modelo Regresión Logística guardado en: /content/drive/My Drive/modelos_smoking/knn.pkl


Regla práctica: El tope recomendado suele ser la raíz cuadrada del número de casos de entrenamiento (N train).
Si tienes 2000 casos de entrenamiento,  
2000≈45.
Si tienes 5000,  
5000≈70.

In [108]:
# ========================================================
# 04_entrenamiento_y_optimizacion - GRADIENT BOOSTING CON VALIDACIÓN
# ========================================================
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import f1_score

# 1. Definición del modelo con regularización
# subsample=0.8 hace que cada árbol vea solo el 80% de los datos (evita memorizar)
# max_depth=4 mantiene los árboles sencillos y rápidos
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    subsample=0.8,
    random_state=42
)

# 2. Entrenamiento
gb_model.fit(X_tr, y_tr)

# 3. Evaluación comparativa (Train vs Test)
pred_tr_gb = gb_model.predict(X_tr)
pred_te_gb = gb_model.predict(X_te)

f1_tr_gb = f1_score(y_tr, pred_tr_gb)
f1_te_gb = f1_score(y_te, pred_te_gb)
diferencia_gb = f1_tr_gb - f1_te_gb

print("--- REPORTE GRADIENT BOOSTING ---")
print(f"F1-Score Entrenamiento: {f1_tr_gb:.4f}")
print(f"F1-Score Test:          {f1_te_gb:.4f}")
print(f"Brecha (Overfitting):   {diferencia_gb:.4f}")

if diferencia_gb < 0.05:
    print("✅ Resultado: Modelo generalizado correctamente.")
else:
    print("⚠️ Nota: Ajusta 'learning_rate' o 'subsample' para reducir el sobreajuste.")

    # 4. Guardado
ruta_modelos = '/content/drive/My Drive/modelos_smoking/'
os.makedirs(ruta_modelos, exist_ok=True)
ruta_gbm = os.path.join(ruta_modelos, 'gboosting.pkl')
joblib.dump(gb_model, ruta_gbm)

print(f"\n🏆 Modelo  guardado en: {ruta_gbm}")

--- REPORTE GRADIENT BOOSTING ---
F1-Score Entrenamiento: 0.7128
F1-Score Test:          0.6907
Brecha (Overfitting):   0.0222
✅ Resultado: Modelo generalizado correctamente.

🏆 Modelo  guardado en: /content/drive/My Drive/modelos_smoking/gboosting.pkl


In [109]:
# ========================================================
# 05_stacking_final_con_evaluacion.ipynb
# ========================================================
from sklearn.ensemble import StackingClassifier, RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

# 1. Split de datos (Manteniendo consistencia)
# (Asegúrate de tener X e y definidos previamente)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Modelos base con parámetros optimizados
base_models = [
    ('rf', RandomForestClassifier(n_estimators=200, max_depth=7, min_samples_leaf=10, random_state=42, n_jobs=-1)),
    ('xgb', XGBClassifier(n_estimators=100, max_depth=4, subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=100, max_depth=4, subsample=0.8, learning_rate=0.1, random_state=42))
]

# 3. Stacking Classifier
stacking_model = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(max_iter=2000),
    cv=5,
    n_jobs=-1
)

# 4. Entrenamiento
print("🚀 Entrenando Stacking Final...")
stacking_model.fit(X_tr, y_tr)

# 5. Evaluación comparativa (Train vs Test)
pred_tr_stack = stacking_model.predict(X_tr)
pred_te_stack = stacking_model.predict(X_te)

f1_tr_stack = f1_score(y_tr, pred_tr_stack)
f1_te_stack = f1_score(y_te, pred_te_stack)
diferencia_stack = f1_tr_stack - f1_te_stack

print(f"--- REPORTE FINAL STACKING ---")
print(f"F1-Score Entrenamiento: {f1_tr_stack:.4f}")
print(f"F1-Score Test:          {f1_te_stack:.4f}")
print(f"Brecha (Overfitting):   {diferencia_stack:.4f}")

# 4. Guardado
ruta_modelos = '/content/drive/My Drive/modelos_smoking/'
os.makedirs(ruta_modelos, exist_ok=True)
ruta_skm = os.path.join(ruta_modelos, 'stackingmod.pkl')
joblib.dump(stacking_model, ruta_skm)

print(f"\n🏆 Modelo Regresión Logística guardado en: {ruta_skm}")

🚀 Entrenando Stacking Final...
--- REPORTE FINAL STACKING ---
F1-Score Entrenamiento: 0.7184
F1-Score Test:          0.6846
Brecha (Overfitting):   0.0338

🏆 Modelo Regresión Logística guardado en: /content/drive/My Drive/modelos_smoking/stackingmod.pkl


In [100]:
# 1. Asegúrate de que f1_tr_stack, f1_te_stack y diferencia_stack
# ya existan en tu memoria antes de ejecutar este bloque.

resultados_comparativos = {
    'Modelo': ['XGBoost', 'Random Forest', 'AdaBoost', 'Gradient Boosting', 'KNN','Regresion logistica', 'Stacking Final'],
    'F1-Score Train': [f1_tr, f1_tr_rf, f1_tr_ada, f1_tr_gb, f1_tr_knn, f1_tr_log, f1_tr_stack], # Sin comillas
    'F1-Score Test':  [f1_te, f1_te_rf, f1_te_ada, f1_te_gb, f1_te_knn_real, f1_te_log, f1_te_stack], # Sin comillas
    'Diferencia':     [diferencia, diferencia_rf, diferencia_ada, diferencia_gb, diferencia_knn, diferencia_log, diferencia_stack] # Sin comillas
}
# 2. Crear DataFrame
df_resultados = pd.DataFrame(resultados_comparativos)

# 3. Ordenar por F1-Score Test de mayor a menor
df_resultados = df_resultados.sort_values(by='F1-Score Train', ascending=False)

# 4. Formato profesional
pd.options.display.float_format = '{:.4f}'.format

print("📊 RESUMEN DE ARQUITECTURAS POR TRAIN")
print("-" * 65)
print(df_resultados.to_string(index=False))
print("-" * 65)

# 2. Crear DataFrame
df_resultados = pd.DataFrame(resultados_comparativos)

# 3. Ordenar por F1-Score Test de mayor a menor
df_resultados = df_resultados.sort_values(by='F1-Score Test', ascending=False)

# 4. Formato profesional
pd.options.display.float_format = '{:.4f}'.format

print("📊 RESUMEN DE ARQUITECTURAS POR TEST")
print("-" * 65)
print(df_resultados.to_string(index=False))
print("-" * 65)

# 2. Crear DataFrame
df_resultados = pd.DataFrame(resultados_comparativos)

# 3. Ordenar por F1-Score Test de mayor a menor
df_resultados = df_resultados.sort_values(by='Diferencia', ascending=True)

# 4. Formato profesional
pd.options.display.float_format = '{:.4f}'.format

print("📊 RESUMEN DE ARQUITECTURAS POR DIFERENCIA")
print("-" * 65)
print(df_resultados.to_string(index=False))
print("-" * 65)

📊 RESUMEN DE ARQUITECTURAS POR TRAIN
-----------------------------------------------------------------
             Modelo  F1-Score Train  F1-Score Test  Diferencia
     Stacking Final          0.7184         0.6846      0.0338
  Gradient Boosting          0.7128         0.6907      0.0222
            XGBoost          0.7118         0.6922      0.0196
      Random Forest          0.6968         0.6779      0.0189
           AdaBoost          0.6778         0.6675      0.0104
                KNN          0.6669         0.6457      0.0212
Regresion logistica          0.6658         0.6650      0.0008
-----------------------------------------------------------------
📊 RESUMEN DE ARQUITECTURAS POR TEST
-----------------------------------------------------------------
             Modelo  F1-Score Train  F1-Score Test  Diferencia
            XGBoost          0.7118         0.6922      0.0196
  Gradient Boosting          0.7128         0.6907      0.0222
     Stacking Final          0.7184 